# momentum-buffer-update — worked example 3: Demonstrate why rebinding b fails and .copy_() succeeds

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `momentum-buffer-update`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The in-place update `b.copy_(mu * b + g)` and the rebinding assignment `b = mu * b + g` look similar but have crucially different behavior in Python. `.copy_()` writes the computed value into the same tensor object, so any other reference to that tensor (such as a list entry) sees the update. Rebinding `b` only changes which object the local variable points to — the list entry still points to the old tensor.

## Worked solution

**Step 1 — store the buffer in a list (simulating optimizer state).** We put the buffer tensor in `buffer_list` so we can check later whether the list entry was mutated.

**Step 2 — rebinding path (broken).** Inside a loop, we do `b = mu * b + g`. The local `b` now points to a newly allocated tensor, but `buffer_list[0]` still points to the original zeros. The next iteration starts from zeros again instead of continuing from the previous step's value.

**Step 3 — in-place path (correct).** Inside a loop, we do `b.copy_(mu * b + g)`. This writes into the existing tensor. `buffer_list[0]` IS the same tensor object as `b`, so the list entry is updated.

**Step 4 — print both trajectories.** After two steps the rebinding trajectory will be `g, g` (same each step, because buffer never accumulates) while the in-place trajectory will be `g, mu*g + g` (accumulating).

In [ ]:
import torch

torch.manual_seed(0)

mu = 0.9
g = torch.tensor([1.0, 2.0, 3.0])  # constant gradient for clarity

# ---- Broken: rebinding ----
buf_rebind = torch.zeros(3)
buffer_list_rebind = [buf_rebind]

print("=== Rebinding (broken) ===")
for step in range(3):
    b = buffer_list_rebind[0]        # grab the list entry
    b = mu * b + g                   # REBIND: b now points to new tensor
    # buffer_list_rebind[0] is STILL the old tensor!
    print(f"  step {step}: list entry = {buffer_list_rebind[0].tolist()}, local b = {b.tolist()}")

# ---- Correct: in-place .copy_() ----
buf_inplace = torch.zeros(3)
buffer_list_inplace = [buf_inplace]

print("\n=== In-place .copy_() (correct) ===")
for step in range(3):
    b = buffer_list_inplace[0]       # grab the list entry (same object)
    b.copy_(mu * b + g)              # MUTATE: the object b points to is updated
    # buffer_list_inplace[0] IS the same object -> it also updated
    print(f"  step {step}: list entry = {buffer_list_inplace[0].tolist()}, local b = {b.tolist()}")

# Verify they diverge after step 1.
print("\nAfter 3 steps:")
print("  Rebinding list entry (still zeros!):", buffer_list_rebind[0].tolist())
print("  In-place list entry (accumulated): ", buffer_list_inplace[0].tolist())